In [0]:
# 01_generate_data – synthetische Bankdaten MIT absichtlichen Qualitätsfehlern
import csv, os, random
from datetime import date, timedelta

random.seed(42)
RAW = "/Volumes/workspace/banking/raw"
for sub in ["customers", "accounts", "transactions"]:
    os.makedirs(f"{RAW}/{sub}", exist_ok=True)

CITIES = ["Düsseldorf", "Köln", "Frankfurt", "München", "Hamburg", "Berlin", "Stuttgart"]
SEGMENTS = ["Privatkunde", "Geschäftskunde", "Premium"]

# ---------- Kunden ----------
customers = []
for i in range(1, 501):
    seg = random.choices(SEGMENTS, weights=[70, 20, 10])[0]
    if random.random() < 0.05:                      # Fehler: Groß/Kleinschreibung + Leerzeichen
        seg = f" {seg.lower()} "
    city = random.choice(CITIES) if random.random() > 0.03 else ""   # Fehler: fehlende Stadt
    birth = date(1950, 1, 1) + timedelta(days=random.randint(0, 20000))
    customers.append([f"C{i:05d}", seg, city, birth.isoformat()])
customers += random.sample(customers, 10)           # Fehler: Duplikate

with open(f"{RAW}/customers/customers.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["customer_id", "segment", "city", "birth_date"])
    w.writerows(customers)

# ---------- Konten ----------
ACC_TYPES = ["Girokonto", "Sparkonto", "Kreditkarte"]
accounts = []
a = 1
for c in customers[:500]:
    for _ in range(random.randint(1, 3)):
        open_d = date(2015, 1, 1) + timedelta(days=random.randint(0, 3650))
        accounts.append([f"A{a:06d}", c[0], random.choice(ACC_TYPES), open_d.isoformat()])
        a += 1

with open(f"{RAW}/accounts/accounts.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["account_id", "customer_id", "account_type", "open_date"])
    w.writerows(accounts)

# ---------- Transaktionen (eine Datei pro Monat) ----------
TX_TYPES = ["Gutschrift", "Lastschrift", "Überweisung", "Kartenzahlung"]
CATEGORIES = ["Lebensmittel", "Miete", "Gehalt", "Reisen", "Elektronik", "Restaurant", "Versicherung"]
CHANNELS = ["Online", "App", "Filiale", "Karte"]
account_ids = [acc[0] for acc in accounts]
tx_id = 1

def fmt_date(d):
    r = random.random()
    if r < 0.85: return d.isoformat()              # 2026-03-15
    if r < 0.95: return d.strftime("%d.%m.%Y")     # Fehler: 15.03.2026
    return d.strftime("%Y/%m/%d")                   # Fehler: 2026/03/15

for month in range(1, 11):                           # Jan–Okt 2026
    rows = []
    for _ in range(3000):
        d = date(2026, month, random.randint(1, 28))
        amount = round(random.lognormvariate(4, 1), 2)
        if random.random() < 0.01:                   # Ausreißer für Auffälligkeits-Analyse
            amount = round(amount * random.randint(20, 50), 2)
        amount_str = f"{amount:.2f}"
        if random.random() < 0.10:                   # Fehler: Komma als Dezimaltrenner
            amount_str = amount_str.replace(".", ",")
        if random.random() < 0.01:                   # Fehler: Betrag fehlt
            amount_str = ""
        currency = random.choices(["EUR", "USD", "CHF", "eur", "XXX"], weights=[85, 6, 4, 4, 1])[0]
        acc = random.choice(account_ids)
        if random.random() < 0.005:                  # Fehler: unbekanntes Konto
            acc = "A999999"
        rows.append([f"T{tx_id:08d}", acc, fmt_date(d), amount_str, currency,
                     random.choice(TX_TYPES), random.choice(CATEGORIES), random.choice(CHANNELS)])
        tx_id += 1
    rows += random.sample(rows, 60)                  # Fehler: ~2 % Duplikate
    with open(f"{RAW}/transactions/transactions_2026_{month:02d}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["transaction_id", "account_id", "booking_date", "amount", "currency",
                    "transaction_type", "merchant_category", "channel"])
        w.writerows(rows)

print(f"Kunden: {len(customers)}, Konten: {len(accounts)}, Transaktionen: {tx_id - 1} (+ Duplikate)")